In [0]:
-- ====================================================================================================================
-- Dimension Date Period
-- ====================================================================================================================
CREATE OR REPLACE VIEW workspace.dimension.dim_date_period AS
WITH CTEDATES AS (
SELECT
      date_sub(current_date(), 1) AS YESTERDAY
    , current_date() AS TODAY
    , date_trunc('week', date_add(current_date(), -7))::DATE AS START_OF_PRIOR_WEEK
    , date_add(date_trunc('week', date_add(current_date(), -7))::DATE, 6) AS END_OF_PRIOR_WEEK
    , date_trunc('month', date_sub(current_date(), 1))::DATE AS START_OF_MONTH
    , date_trunc('month', add_months(date_sub(current_date(), 1), -1))::DATE AS START_OF_PRIOR_MONTH
    , last_day(add_months(date_sub(current_date(), 1), -1)) AS END_OF_PRIOR_MONTH
    , CASE WHEN month(date_sub(current_date(), 1)) >= 7 THEN make_date(year(date_sub(current_date(), 1)), 7, 1)
      ELSE make_date(year(date_sub(current_date(), 1)) - 1, 7, 1)
      END AS START_OF_FIN_YEAR
    , CASE WHEN month(date_sub(current_date(), 1)) < 7 THEN make_date(year(date_sub(current_date(), 1)), 6, 30) 
      ELSE make_date(year(date_sub(current_date(), 1)) + 1, 6, 30)
      END AS END_OF_FIN_YEAR
    , add_months(START_OF_FIN_YEAR, -12) AS START_OF_PRIOR_FY
    , add_months(END_OF_FIN_YEAR, -12) AS END_OF_PRIOR_FY
    , make_date(year(date_sub(current_date(), 1)), 1, 1) AS START_OF_YEAR
)
--select * from CTEDATES
-- ====================================================
-- Yesterday
-- ====================================================
SELECT
    DATE_ID
  , 'Yesterday' AS DATE_PERIOD
  , 1 AS PERIOD_ORDER
FROM workspace.dimension.dim_date
WHERE DATE = (SELECT YESTERDAY FROM CTEDATES)
UNION ALL
-- ====================================================
-- Prior Week
-- ====================================================
SELECT
    DATE_ID
  , 'Prior Week' AS DATE_PERIOD
  , 2 AS PERIOD_ORDER
FROM workspace.dimension.dim_date
WHERE DATE BETWEEN (SELECT START_OF_PRIOR_WEEK FROM CTEDATES)
                   AND (SELECT END_OF_PRIOR_WEEK FROM CTEDATES)
UNION ALL
-- ====================================================
-- MTD
-- ====================================================
SELECT
    DATE_ID
  , 'MTD' AS DATE_PERIOD
  , 3 AS PERIOD_ORDER
FROM workspace.dimension.dim_date
WHERE DATE BETWEEN (SELECT START_OF_MONTH FROM CTEDATES)
                   AND (SELECT YESTERDAY FROM CTEDATES)
UNION ALL
-- ====================================================
-- Prior Month
-- ====================================================
SELECT
    DATE_ID
  , 'Prior Month' AS DATE_PERIOD
  , 4 AS PERIOD_ORDER
FROM workspace.dimension.dim_date dd
WHERE DATE BETWEEN (SELECT START_OF_PRIOR_MONTH FROM CTEDATES)
                   AND (SELECT END_OF_PRIOR_MONTH FROM CTEDATES)
UNION ALL
-- ====================================================
-- FYTD
-- ====================================================
SELECT
    DATE_ID
  , 'FYTD' AS DATE_PERIOD
  , 5 AS PERIOD_ORDER
FROM workspace.dimension.dim_date
WHERE DATE BETWEEN (SELECT START_OF_FIN_YEAR FROM CTEDATES)
                   AND (SELECT END_OF_FIN_YEAR FROM CTEDATES)
UNION ALL
-- ====================================================
-- Prior Fin Year
-- ====================================================
SELECT
    DATE_ID
  , 'Prior Fin Year' AS DATE_PERIOD
  , 6 AS PERIOD_ORDER
FROM workspace.dimension.dim_date 
WHERE DATE BETWEEN (SELECT START_OF_PRIOR_FY FROM CTEDATES)
                   AND (SELECT END_OF_PRIOR_FY FROM CTEDATES)
UNION ALL
-- ====================================================
-- YTD
-- ====================================================
SELECT
    DATE_ID
  , 'YTD' AS DATE_PERIOD
  , 7 AS PERIOD_ORDER
FROM workspace.dimension.dim_date
WHERE DATE BETWEEN (SELECT START_OF_YEAR FROM CTEDATES)
                   AND (SELECT YESTERDAY FROM CTEDATES)
UNION ALL
-- ====================================================
-- Custom (Make sure to clear the date filter before switching back)
-- ====================================================
SELECT
    DATE_ID
  , 'Custom' AS DATE_PERIOD
  , 8 AS PERIOD_ORDER
FROM workspace.dimension.dim_date dd
WHERE DATE <= (SELECT YESTERDAY FROM CTEDATES)